In [1]:
import duckdb
import pandas as pd
import numpy as np
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from networks.feed_forward_tabular import NeuralModel
from networks.feed_forward_tabular import build_dataloaders

con = duckdb.connect('../capillary.db')

df = con.execute(""" SELECT row_id, value, label, albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm,set FROM protein_data WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL""").df()

con.close()


print(f"Antal fall med M-komponent:{(df['label'] == 1).sum()}")
print(f"Antal fall utan M-komponent:{(df['label'] == 0).sum()}")


train_rows = df[df['set'] == 'train']
val_rows = df[df['set'] == 'val']
test_rows = df[df['set'] == 'test']

drop_indices = train_rows[train_rows['label'] == 0].sample(frac=0.7).index


train_rows = train_rows.drop(drop_indices)
train_rows = train_rows[train_rows['label'].isin([0,1])]
val_rows = val_rows[val_rows['label'].isin([0,1])]

print(f"Antal utan m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 0])}")
print(f"Antal med m-komponent i träningsdatan: {len(train_rows[train_rows['label'] == 1])}")

network = NeuralModel()
network.reset_weights()
cnn_train_dl, cnn_val_dl, _ = build_dataloaders(train_rows, val_rows, val_rows)
network.retrain(cnn_train_dl,cnn_val_dl,patience=15)

Antal fall med M-komponent:2942
Antal fall utan M-komponent:69882
Antal utan m-komponent i träningsdatan: 17903
Antal med m-komponent i träningsdatan: 2553
Total parameters: 251,970
  -> ny bästa modell sparad till ../models/feed_forward_tabular.pth
Epoch   0 | train: 0.8500 | val: 0.5304 | acc: 85.94% | AUC: 0.775  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forward_tabular.pth
Epoch   1 | train: 0.5607 | val: 0.5522 | acc: 77.12% | AUC: 0.840  | LR: 0.001
Epoch   2 | train: 0.5090 | val: 0.5680 | acc: 70.89% | AUC: 0.809  | LR: 0.001
Epoch   3 | train: 0.4802 | val: 0.3681 | acc: 96.99% | AUC: 0.785  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forward_tabular.pth
Epoch   4 | train: 0.4605 | val: 0.3665 | acc: 90.17% | AUC: 0.913  | LR: 0.001
Epoch   5 | train: 0.4424 | val: 0.3332 | acc: 94.98% | AUC: 0.895  | LR: 0.001
Epoch   6 | train: 0.4229 | val: 0.5723 | acc: 72.19% | AUC: 0.888  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forwar

In [2]:
con = duckdb.connect('../capillary.db')

df = con.execute(""" SELECT row_id, value, label, albumin,antitrypsin,orosomukoid,haptoglobin,crp,igg,iga,igm,set FROM protein_data WHERE value IS NOT NULL
                 AND observation_nr = 1
                 AND analysis IS NOT NULL
                 AND protein_value IS NOT NULL""").df()

con.close()


k_fold_rows = df[df['set'].isin(['train','val']) ].copy()
k_fold_rows = k_fold_rows[k_fold_rows['label'].isin([0,1])]
drop_indices = k_fold_rows[k_fold_rows['label'] == 0].sample(frac=0.7).index
k_fold_rows = k_fold_rows.drop(drop_indices)
print(f"Totalt antal utan m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 0])}")
print(f"Totalt antal med m-komponent i K-fold poolen: {len(k_fold_rows[k_fold_rows['label'] == 1])}")

test_rows  = df[df['set'] == 'test']


network = NeuralModel()
network.retrain_with_k_fold(k_fold_rows)

Totalt antal utan m-komponent i K-fold poolen: 18848
Totalt antal med m-komponent i K-fold poolen: 2692
Total parameters: 251,970
--- Startar 10-Fold Cross Validation ---

 FOLD 1/10
  -> ny bästa modell sparad till ../models/feed_forward_tabular_fold1.pth
Epoch   0 | train: 1.0903 | val: 0.6182 | acc: 68.14% | AUC: 0.699  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forward_tabular_fold1.pth
Epoch   1 | train: 0.5453 | val: 0.5353 | acc: 86.34% | AUC: 0.779  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forward_tabular_fold1.pth
Epoch   2 | train: 0.5262 | val: 0.5199 | acc: 72.78% | AUC: 0.835  | LR: 0.001
Epoch   3 | train: 0.4764 | val: 0.5313 | acc: 92.15% | AUC: 0.820  | LR: 0.001
  -> ny bästa modell sparad till ../models/feed_forward_tabular_fold1.pth
Epoch   4 | train: 0.4395 | val: 0.4697 | acc: 90.34% | AUC: 0.857  | LR: 0.001
Epoch   5 | train: 0.4859 | val: 0.4985 | acc: 89.32% | AUC: 0.803  | LR: 0.001
Epoch   6 | train: 0.4319 | val: 0.5688 |

,fold,train_loss,val_loss,val_accuracy,val_auc,val_spec,val_sens,tn,fp,fn,tp
0,1,0.157416,0.243230,94.101254,0.963162,0.952735,0.859259,1794,89,38,232
1,2,0.195912,0.208345,94.240595,0.973842,0.954859,0.855556,1798,85,39,231
2,3,0.155395,0.217295,94.939647,0.962155,0.964987,0.840149,1819,66,43,226
3,4,0.192577,0.239128,94.983744,0.962180,0.962314,0.862454,1813,71,37,232
4,5,0.141843,0.263361,93.268338,0.959705,0.943236,0.858736,1778,107,38,231
5,6,0.181835,0.187521,95.864312,0.965648,0.972384,0.862454,1831,52,37,232
6,7,0.186392,0.217639,94.333488,0.961261,0.962845,0.806691,1814,70,52,217
7,8,0.219473,0.285681,93.633829,0.950226,0.952204,0.825279,1793,90,47,222
8,9,0.161569,0.237828,94.289694,0.963803,0.953316,0.869888,1797,88,35,234
9,10,0.159384,0.193768,94.377323,0.968499,0.954328,0.869888,1797,86,35,234


In [3]:
from functions.evaluation import evaluate


test_rows = test_rows[test_rows['label'].isin([0,1])]
test_rows['cnn_probability'] = test_rows['probability'].copy()
result = network.predict(test_rows)
_ = evaluate(result)

KeyError: 'probability'